# Module 6 — Machine Learning on the gold layer

## Story beat: "Predict, don't just report"

Modules 1–5 gave Contoso clean **gold** tables and live telemetry. Now the business asks a forward-looking question: *"What will a store sell tomorrow?"* We train a model on `gold.sales_by_store_day`, track it with **MLflow** (built into Fabric), **register** it in the workspace model registry, **score** it, and write predictions back to the lakehouse as `gold.sales_predictions`.

Everything stays in **OneLake / Fabric** — no data leaves, no separate ML platform to wire up.

---

## What this notebook does

| Step | Action |
| --- | --- |
| 1 | Load `gold.sales_by_store_day` |
| 2 | Feature engineering (day-of-week, region, units, transactions) |
| 3 | Train a regressor + **MLflow autolog** + **register** the model |
| 4 | Score and write `gold.sales_predictions` back to the lakehouse |

> **Prereq:** Module 1 produced the gold tables, and this notebook's **default lakehouse = `lh_retail`** (bound by `run.ps1`).

> **Presenter note:** *"Fabric is a full data-science platform too — MLflow experiments, a model registry, and SynapseML are built in. The model trains on the same gold tables the BI report uses."*

## Step 1 — Load the gold sales table

`gold.sales_by_store_day` is one row per store per day (net_sales, units, transactions, region). Small enough to pull into pandas for scikit-learn.

In [ ]:
import pandas as pd

pdf = spark.table("gold.sales_by_store_day").toPandas()
print("rows:", len(pdf))
pdf.head()

## Step 2 — Feature engineering

- **`dow`** — day of week from `sale_date` (sales have weekly seasonality).
- **`region`** — one-hot encoded.
- **`units`, `transactions`** — same-day drivers.
- **Target = `net_sales`.**

We split chronologically-ish with a random hold-out for a quick demo metric.

In [ ]:
from sklearn.model_selection import train_test_split

pdf["sale_date"] = pd.to_datetime(pdf["sale_date"])
pdf["dow"] = pdf["sale_date"].dt.dayofweek
feat = pd.get_dummies(pdf[["store_id", "dow", "units", "transactions", "region"]], columns=["region"])
X = feat
y = pdf["net_sales"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("features:", list(X.columns))
print("train/test:", len(X_train), len(X_test))

## Step 3 — Train, track with MLflow, and register

MLflow is **built into Fabric** — `autolog()` captures params, metrics, and the model automatically. The run shows up under the workspace **Experiments**; `register_model` publishes it to the **model registry** as a versioned item.

> **Watch in the UI:** after this cell, open the `retail-sales-forecast` experiment and the `retail_sales_forecaster` registered model in the workspace.

In [ ]:
import mlflow
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

mlflow.set_experiment("retail-sales-forecast")
mlflow.sklearn.autolog(registered_model_name="retail_sales_forecaster")

with mlflow.start_run() as run:
    model = RandomForestRegressor(n_estimators=120, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    print(f"run_id={run.info.run_id}  MAE={mae:,.2f}  R2={r2:.3f}")

## Step 4 — Score and write predictions back to the lakehouse

We score every row, attach the prediction + error, and write `gold.sales_predictions` as Delta — so Power BI / the warehouse / a Data Agent can all consume the model output, **zero copy**, like any other gold table.

In [ ]:
scored = pdf[["sale_date", "store_id", "region", "net_sales"]].copy()
scored["predicted_net_sales"] = model.predict(X).round(2)
scored["abs_error"] = (scored["net_sales"] - scored["predicted_net_sales"]).abs().round(2)
scored["sale_date"] = scored["sale_date"].dt.date.astype(str)

sdf = spark.createDataFrame(scored)
sdf.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold.sales_predictions")
print("wrote gold.sales_predictions:", sdf.count(), "rows")
display(spark.sql("SELECT * FROM gold.sales_predictions ORDER BY abs_error DESC LIMIT 10"))

---

## ✅ Success looks like

- An MLflow run under experiment **`retail-sales-forecast`** with `mae` / `r2` metrics.
- A registered model **`retail_sales_forecaster`** in the workspace model registry.
- A new **`gold.sales_predictions`** Delta table.

**Talking points:**
- Same OneLake gold tables feed BI **and** ML — one estate.
- MLflow tracking + registry are native; no separate ML service.
- Predictions become a **gold data product** any engine (or a Module 9 Data Agent) can query.

**Next:** Module 7 — orchestrate & govern (you could schedule this notebook in a pipeline).